In [ ]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # Name of the GPU


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF



In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels): #input channels(1 for grayscale and 3 for rgb) and output channels(no. of feature maps)
        super(DoubleConv, self).__init__()         #callin nn.module constructor
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),  #3-3*3 kernel size, 1 stride, 1 padding, bias false as batch norm introduces beta bias anyway
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),  #inplace saves memory, input only modified as output
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):  #passing input x through convolution and returning output
        return self.conv(x)

In [ ]:
class UNET(nn.Module):
    def __init__(
            self, in_channels=3, out_channels=1, features=[64, 128, 256, 512],):  #input is rgb, output is a binary mask, features are number of filters at each level
        super(UNET, self).__init__()       #calling nn.module constructor
        self.ups = nn.ModuleList()         #Stores upsampling layers
        #module list is layered python list
        self.downs = nn.ModuleList()       #Stores downsampling layers
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) 
        #2*2,s=2 maxpool defined as standalone layer as it doesn't have learnable parameters

        # Downsampling layers
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature)) 
            #3,64; 64,128; 128,256; 256,512
            in_channels = feature

        # Upsampling layers
        for feature in reversed(features): #going big to small
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2,))  #transpose decreases no. of feature channels to half
            self.ups.append(DoubleConv(feature*2, feature)) #but half feature channels joined with skip connection from encoder again gives feature*2 input to doble convode

        self.bottleneck = DoubleConv(features[-1], features[-1]*2) #deepest layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        # last conv 1*1 layer
    
    def forward(self, x): #forward in unet for handling down/upsampling and skip connections
        skip_connections = []   #list will be used to store skip connections of different layers

        for down in self.downs:  #so down is each double conv
            x = down(x)  #x input to double conv (in pytorch, automatically calls forward function, overrides call method)
            skip_connections.append(x) #x is now the output
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]  #reversing list of skip connections

        for idx in range(0, len(self.ups), 2):  #step 2 because 1 transpose+ 1 double conv (goes transpose to transpose)
            x = self.ups[idx](x)  #returns transposed(half feature maps)
            skip_connection = skip_connections[idx//2] #retrieves corresponding skip conn

            if x.shape != skip_connection.shape:
                #difference in size due to rounding of deimals in half and double
                x = TF.resize(x, size=skip_connection.shape[2:]) #2: considers only height and weight

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip) #double conv output with concat as input 

        return self.final_conv(x)

In [ ]:
import os

data_path = "/kaggle/input/"
print(os.listdir(data_path)) 

In [ ]:
pip install pycocotools


In [ ]:
import os

base_path = "/kaggle/input"
print(os.listdir(base_path))


In [ ]:
from pycocotools.coco import COCO
import os
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as transforms


annotation_path = "/kaggle/input/annotation.json"
coco = COCO(annotation_path)


image_ids = list(coco.imgs.keys())


In [ ]:
image_dir = "/kaggle/input/satellite-segmentation/train/images"

In [ ]:
def load_image_and_mask(image_id, coco, image_dir):
    
    img_info = coco.loadImgs(image_id)[0]
    img_path = os.path.join(image_dir, img_info['file_name'])

    if not os.path.exists(img_path):
        raise FileNotFoundError(f"Image file not found: {img_path}")

    img = Image.open(img_path).convert('RGB')

   
    ann_ids = coco.getAnnIds(imgIds=image_id)
    anns = coco.loadAnns(ann_ids)

    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    if not anns:
        print(f"Warning: No annotations found for image ID {image_id}. Returning an empty mask.")

    for ann in anns:
        mask = np.maximum(coco.annToMask(ann), mask)

    mask = mask.astype(np.float32)  # Normalize for PyTorch

    return img, mask



In [ ]:
from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
from PIL import Image

def transform_image_and_mask(img, mask, transform):
    img = transform(img) 

    # Convert mask (NumPy array) to PIL Image before resizing
    mask = Image.fromarray(mask)
    mask = resize(mask, (256, 256), interpolation=InterpolationMode.NEAREST)
    
    mask = torch.tensor(np.array(mask), dtype=torch.float32)

    mask = (mask > 0.5).float()  # Ensures strict binary values


    return img, mask


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class COCOSegmentationDataset(Dataset):
    def __init__(self, coco, image_dir, transform=None, ids=None):
        self.coco = coco
        self.image_dir = image_dir
        self.transform = transform
        self.image_ids = ids.copy() if ids is not None else list(coco.imgs.keys())

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        try:
            img, mask = load_image_and_mask(img_id, self.coco, self.image_dir)
        except Exception as e:
            print(f"Warning: Failed to load image ID {img_id}: {e}")
            img = Image.new('RGB', (256, 256), color=(0, 0, 0))  # Placeholder 
            mask = np.zeros((256, 256), dtype=np.uint8)

        if self.transform:
            img, mask = transform_image_and_mask(img, mask, self.transform)

        #mask should be a PyTorch tensor with a single channel
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)


        return img, mask

    def __len__(self):
        return len(self.image_ids)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def visualize_image_mask_overlay(image_id, coco, image_dir):
    img, mask = load_image_and_mask(image_id, coco, image_dir)
    img_array = np.array(img)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title("Image")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Mask")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(img_array)
    plt.imshow(mask, cmap='jet', alpha=0.5) 
    plt.title("Overlay")
    plt.axis('off')

    plt.show()
visualize_image_mask_overlay(image_id=image_ids[0], coco=coco, image_dir="/kaggle/input/train/images")


In [ ]:
from sklearn.model_selection import train_test_split
train_ids, remaining_ids = train_test_split(image_ids, test_size=0.3, random_state=42)  # 70% train, 30% remaining
val_ids, test_ids = train_test_split(remaining_ids, test_size=0.5, random_state=42)  # 50% of remaining for validation and 50% for test
print(f"Training Set Size: {len(train_ids)}")
print(f"Validation Set Size: {len(val_ids)}")
print(f"Test Set Size: {len(test_ids)}")


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize images to 256x256
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])# Convert images to PyTorch tensors
])


In [ ]:

image_dir = "/kaggle/input/train/images"  
train_dataset = COCOSegmentationDataset(coco, image_dir, transform=transform, ids=train_ids)
val_dataset = COCOSegmentationDataset(coco, image_dir, transform=transform, ids=val_ids)
test_dataset= COCOSegmentationDataset(coco, image_dir, transform=transform, ids=test_ids)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNET(in_channels=3, out_channels=1).to(device)

criterion = torch.nn.BCEWithLogitsLoss()  # For binary segmentation (0/1 masks)
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
criterion = torch.nn.BCEWithLogitsLoss().to(device)


In [ ]:

import torch

def dice_coefficient(preds, targets, threshold=0.5, eps=1e-7):
    
    preds = torch.sigmoid(preds) > threshold 
    targets = targets.bool()

    intersection = (preds & targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))

    dice = (2. * intersection + eps) / (union + eps)  
    return dice.mean().item() 

def iou_score(preds, targets, threshold=0.5, eps=1e-7):
    
    preds = torch.sigmoid(preds) > threshold
    targets = targets.bool()

    intersection = (preds & targets).sum(dim=(1, 2, 3))
    union = (preds | targets).sum(dim=(1, 2, 3))

    iou = (intersection + eps) / (union + eps)  
    return iou.mean().item() 


In [ ]:
def validate_fn(loader, model, loss_fn, device, max_batches=None):
    """Validate the model and return average loss, Dice, and IoU"""
    model.eval()  # Ensure model is in evaluation mode
    total_loss = 0
    total_dice = 0
    total_iou = 0
    num_batches = 0

    with torch.no_grad():  # No gradients for validation
        for batch_idx, (imgs, masks) in enumerate(loader):
            imgs, masks = imgs.to(device), masks.to(device).float()


            preds = model(imgs)
            loss = loss_fn(preds, masks.float())

            # Compute Dice & IoU
            dice = dice_coefficient(preds, masks)
            iou = iou_score(preds, masks)

            # Accumulate metrics
            total_loss += loss.item()
            total_dice += dice
            total_iou += iou
            num_batches += 1

            if max_batches and num_batches >= max_batches:
                break

    # Compute averages to avoid division by zero
    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    avg_dice = total_dice / num_batches if num_batches > 0 else 0
    avg_iou = total_iou / num_batches if num_batches > 0 else 0

    return avg_loss, avg_dice, avg_iou


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch

In [ ]:
def train_fn(train_loader, val_loader, model, optimizer, loss_fn, device, val_interval=10):
    """Train the model and validate every 'val_interval' batches"""
    model.train()
    total_loss = 0

    for batch_idx, (imgs, masks) in enumerate(train_loader):
        model.train()  # Ensure model is in training mode

        imgs, masks = imgs.to(device), masks.to(device).float()


        # Forward pass
        preds = model(imgs)
        loss = loss_fn(preds, masks.float())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        

        # Print training metrics every 10 batches
        if (batch_idx + 1) % 10 == 0:
            print(f"Train Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        # Perform validation every 'val_interval' batches
        if (batch_idx + 1) % val_interval == 0:
            val_loss, avg_dice, avg_iou = validate_fn(val_loader, model, loss_fn, device, max_batches=10)
            print(f"Validation after {batch_idx+1} batches: Loss: {val_loss:.4f}, Dice: {avg_dice:.4f}, IoU: {avg_iou:.4f}")

    return total_loss / len(train_loader)




In [ ]:
num_epochs = 3

# Initialize total sums for averaging
total_train_loss = 0
total_val_loss = 0
total_val_dice = 0
total_val_iou = 0

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}] -------------------------")

    train_loss = train_fn(train_loader, val_loader, model, optimizer, criterion, device)
    val_loss, val_dice, val_iou = validate_fn(val_loader, model, criterion, device)

    # Accumulate values for averaging
    total_train_loss += train_loss
    total_val_loss += val_loss
    total_val_dice += val_dice
    total_val_iou += val_iou

    # Print metrics for current epoch
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")


avg_train_loss = total_train_loss / num_epochs
avg_val_loss = total_val_loss / num_epochs
avg_val_dice = total_val_dice / num_epochs
avg_val_iou = total_val_iou / num_epochs


print("\nFinal Averaged Metrics Across All Epochs:")
print(f"Average Train Loss: {avg_train_loss:.4f}")
print(f"Average Validation Loss: {avg_val_loss:.4f}")
print(f"Average Dice: {avg_val_dice:.4f}")
print(f"Average IoU: {avg_val_iou:.4f}")


In [ ]:
def test_fn(test_loader, model, loss_fn, device):
    """Test the model on the test set and return average loss, Dice, and IoU"""
    model.eval() 
    total_loss = 0
    total_dice = 0
    total_iou = 0
    num_batches = 0

    with torch.no_grad(): 
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device).float()

            preds = model(imgs)
            loss = loss_fn(preds, masks.float())

            dice = dice_coefficient(preds, masks)
            iou = iou_score(preds, masks)

            total_loss += loss.item()
            total_dice += dice
            total_iou += iou
            num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    avg_dice = total_dice / num_batches if num_batches > 0 else 0
    avg_iou = total_iou / num_batches if num_batches > 0 else 0

    return avg_loss, avg_dice, avg_iou




In [ ]:
# Run Testing
test_loss, test_dice, test_iou = test_fn(test_loader, model, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Dice: {test_dice:.4f}, IoU: {test_iou:.4f}")

In [ ]:
import torch
import matplotlib.pyplot as plt

def predict_and_visualize(model, image, device):
    """Predicts and visualizes the segmentation mask for a single image."""
    model.eval()  # Set model to evaluation mode
    
    with torch.no_grad():  
        image = image.to(device).unsqueeze(0)  
        pred_mask = torch.sigmoid(model(image))  
        pred_mask = (pred_mask > 0.5).float() 
        pred_mask = pred_mask.squeeze(0).cpu().numpy()  
    
    # Convert input image to numpy (assuming it's in [0,1] range)
    image = image.squeeze(0).permute(1, 2, 0).cpu().numpy()

    # Plot original image and predicted mask
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    axes[0].imshow(image)  # Original Image
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(pred_mask[0], cmap="gray")  # Predicted Mask
    axes[1].set_title("Predicted Mask")
    axes[1].axis("off")

    plt.show()

    return pred_mask


In [ ]:
sample_image, _ = test_dataset[0]
pred_mask = predict_and_visualize(model, sample_image, device)